# PHẦN 5. PHÂN TÍCH KỊCH BẢN (SCENARIO ANALYSIS)
## Phân tích tác động của lãi suất và lạm phát đến lợi suất cổ phiếu Microsoft (MSFT)
**Cổ phiếu:** Microsoft Corporation (MSFT)  
**Thời gian nghiên cứu:** 01/2021 – 08/2026  
**Tần suất dữ liệu:** Monthly  
### Mục tiêu
Sử dụng mô hình MLR đã xây dựng để dự báo lợi suất MSFT dưới các kịch bản lãi suất khác nhau, trong khi giữ nguyên tỷ lệ lạm phát ở mức thực tế gần nhất.
### Các biến
- **Y:** Monthly Return của MSFT  
- **X1:** Federal Funds Effective Rate (Interest Rate)  
- **X2:** Inflation Rate (CPI YoY)

## 5.1. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
print("Import thư viện thành công!")

## 5.2. Tải và chuẩn bị dữ liệu
Dữ liệu được đọc từ file CSV đã xử lý ở Phần 1. Sau đó tính Monthly Return và xây dựng lại mô hình OLS để phục vụ dự báo kịch bản.

In [ ]:
df = pd.read_csv("../data/msft_macro_2021_2026.csv")

df.head()

In [ ]:
df['Monthly_Return'] = df['MSFT_Price'].pct_change() * 100

df_model = df[['Monthly_Return', 'Interest_Rate', 'Inflation_Rate']].dropna()

df_model.head()

In [ ]:
df.tail()

## 5.3. Xây dựng lại mô hình OLS
Mô hình hồi quy tuyến tính đa biến (OLS) được xây dựng lại để lấy tham số phục vụ cho việc dự báo kịch bản.

In [ ]:
X = df_model[['Interest_Rate', 'Inflation_Rate']]
X = sm.add_constant(X)

y = df_model['Monthly_Return']

model = sm.OLS(y, X).fit()

print(model.summary())

## 5.4. Thiết lập các kịch bản lãi suất
Kịch bản được xây dựng dựa trên dữ liệu thực tế của tháng gần nhất trong bộ dữ liệu.  
Lạm phát được giữ nguyên ở mức thực tế (ceteris paribus), chỉ thay đổi lãi suất theo 3 mức:
- **Base:** Lãi suất hiện tại  
- **Rate Increase +0.5pp:** Tăng thêm 0.5 điểm phần trăm  
- **Rate Increase +1.0pp:** Tăng thêm 1.0 điểm phần trăm

In [ ]:
latest = df.iloc[-1]

print("Tháng cuối cùng:", latest['Date'])
print("MSFT Price:", latest['MSFT_Price'])
print("Interest Rate:", latest['Interest_Rate'])
print("Inflation Rate:", latest['Inflation_Rate'])

In [ ]:
latest_rate = latest['Interest_Rate']
latest_inflation = latest['Inflation_Rate']

scenario = pd.DataFrame({
    'Scenario': [
        'Base',
        'Rate Increase +0.5pp',
        'Rate Increase +1.0pp'
    ],
    'Interest_Rate': [
        latest_rate,
        latest_rate + 0.5,
        latest_rate + 1.0
    ],
    'Inflation_Rate': [
        latest_inflation,
        latest_inflation,
        latest_inflation
    ]
})

scenario

## 5.5. Dự báo lợi suất theo từng kịch bản
Sử dụng mô hình OLS để tính lợi suất dự báo và khoảng tin cậy 95% (CI) cùng khoảng dự báo 95% (PI) cho từng kịch bản.

In [ ]:
X_scenario = sm.add_constant(
    scenario[['Interest_Rate', 'Inflation_Rate']],
    has_constant='add'
)

scenario['Predicted_Return'] = model.predict(X_scenario)

scenario

In [ ]:
prediction = model.get_prediction(X_scenario)

prediction_summary = prediction.summary_frame(alpha=0.05)

scenario['CI_Lower'] = prediction_summary['mean_ci_lower'].values
scenario['CI_Upper'] = prediction_summary['mean_ci_upper'].values

scenario['PI_Lower'] = prediction_summary['obs_ci_lower'].values
scenario['PI_Upper'] = prediction_summary['obs_ci_upper'].values

scenario

## 5.6. Tổng hợp kết quả
Bảng kết quả tổng hợp lợi suất dự báo, mức thay đổi so với kịch bản cơ sở, khoảng tin cậy và khoảng dự báo cho từng kịch bản.

In [ ]:
result = scenario.copy()

result['Interest_Rate'] = result['Interest_Rate'].round(2)
result['Inflation_Rate'] = result['Inflation_Rate'].round(2)
result['Predicted_Return'] = result['Predicted_Return'].round(4)
result['CI_Lower'] = result['CI_Lower'].round(4)
result['CI_Upper'] = result['CI_Upper'].round(4)
result['PI_Lower'] = result['PI_Lower'].round(4)
result['PI_Upper'] = result['PI_Upper'].round(4)

result

In [ ]:
base_return = result.loc[
    result['Scenario'] == 'Base',
    'Predicted_Return'
].iloc[0]

result['Change_vs_Base'] = (
    result['Predicted_Return'] - base_return
).round(4)

result

In [ ]:
report_table = result[
    [
        'Scenario',
        'Interest_Rate',
        'Inflation_Rate',
        'Predicted_Return',
        'Change_vs_Base',
        'CI_Lower',
        'CI_Upper',
        'PI_Lower',
        'PI_Upper'
    ]
].copy()

report_table

In [ ]:
print("SCENARIO ANALYSIS - MSFT")
print("=" * 50)
print(result.to_string(index=False))

## 5.7. Trực quan hóa kết quả
### 5.7.1. Biểu đồ lợi suất dự báo theo kịch bản

In [ ]:
plt.figure(figsize=(10, 6))

x = range(len(result))
y = result['Predicted_Return']

plt.plot(
    x,
    y,
    marker='o',
    linewidth=2.5,
    markersize=8,
    color='steelblue'
)

for i, value in enumerate(y):
    plt.annotate(
        f'{value:.4f}%',
        (i, value),
        xytext=(0, 10),
        textcoords='offset points',
        ha='center',
        fontsize=10
    )

plt.axhline(
    y=0,
    color='gray',
    linestyle='--',
    linewidth=1
)

plt.xticks(
    x,
    [
        'Base
(3.63%)',
        '+0.5pp
(4.13%)',
        '+1.0pp
(4.63%)'
    ]
)

plt.xlabel('Interest Rate Scenario', fontsize=11)
plt.ylabel('Predicted Monthly Return (%)', fontsize=11)

plt.title(
    'MSFT Predicted Monthly Return under Interest Rate Scenarios',
    fontsize=14,
    fontweight='bold',
    pad=15
)

plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

### 5.7.2. Biểu đồ lợi suất dự báo với khoảng dự báo 95% (Prediction Interval)

In [ ]:
plt.figure(figsize=(10, 6))

x = np.arange(len(result))
y = result['Predicted_Return']
lower = result['PI_Lower']
upper = result['PI_Upper']

plt.errorbar(
    x,
    y,
    yerr=[
        y - lower,
        upper - y
    ],
    fmt='o-',
    capsize=7,
    linewidth=2,
    markersize=8,
    color='steelblue',
    ecolor='tomato',
    label='Predicted Return ± 95% PI'
)

for i, value in enumerate(y):
    plt.annotate(
        f'{value:.4f}%',
        (i, value),
        xytext=(0, 12),
        textcoords='offset points',
        ha='center',
        fontsize=10
    )

plt.axhline(y=0, color='gray', linestyle='--', linewidth=1)

plt.xticks(
    x,
    [
        'Base
(3.63%)',
        '+0.5pp
(4.13%)',
        '+1.0pp
(4.63%)'
    ]
)

plt.xlabel('Interest Rate Scenario', fontsize=11)
plt.ylabel('Monthly Return (%)', fontsize=11)

plt.title(
    'MSFT Predicted Return and 95% Prediction Interval',
    fontsize=14,
    fontweight='bold',
    pad=15
)

plt.legend(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## 5.8. Nhận xét kết quả Scenario Analysis

Phân tích kịch bản kiểm tra xem lợi suất tháng dự báo của MSFT thay đổi như thế nào khi lãi suất thay đổi, trong khi tỷ lệ lạm phát được giữ cố định ở mức thực tế gần nhất (~3.35%).

**Kết quả chính:**
- Ở kịch bản **Base** (lãi suất 3.63%), mô hình dự báo lợi suất tháng là **1.9254%**.
- Khi lãi suất tăng thêm **+0.5pp** (4.13%), lợi suất dự báo tăng nhẹ lên **1.9438%**.
- Khi lãi suất tăng thêm **+1.0pp** (4.63%), lợi suất dự báo tăng lên **1.9622%**.

**Lưu ý quan trọng:**
Mặc dù mô hình cho thấy lợi suất dự báo tăng nhẹ khi lãi suất tăng, kết quả này **không được** hiểu là bằng chứng rằng lãi suất cao hơn sẽ làm tăng lợi suất MSFT. Hệ số hồi quy của biến Interest Rate trong mô hình **không có ý nghĩa thống kê**. Do đó, kết quả scenario analysis nên được hiểu là phản ứng có điều kiện của mô hình hồi quy, chứ không phải là dự báo đáng tin cậy về lợi suất trong tương lai.

Khoảng dự báo 95% (PI) cũng tương đối rộng ở tất cả các kịch bản, cho thấy **độ không chắc chắn cao** khi dự báo lợi suất của một cổ phiếu riêng lẻ.

**Kết luận:** Kết quả scenario analysis chủ yếu minh họa độ nhạy của mô hình với giả định lãi suất, không nên sử dụng như dự báo chính xác về hiệu suất trong tương lai của MSFT.